In [1]:
from plotly.io import show

from skfolio import Population, RiskMeasure
from skfolio.datasets import load_sp500_dataset
from skfolio.distribution import StudentTCopula, VineCopula, compute_pseudo_observations
from skfolio.optimization import MeanRisk
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()
prices = prices[["AMD", "BAC", "HD", "JPM", "LLY", "CVX"]]
X = prices_to_returns(prices)
print(X.tail())

                 AMD       BAC        HD       JPM       LLY       CVX
Date                                                                  
2022-12-21  0.040430  0.015223  0.014358  0.011248  0.023275  0.011758
2022-12-22 -0.056442 -0.008848 -0.010146 -0.011355 -0.007339 -0.014998
2022-12-23  0.010335  0.002443  0.008257  0.004749  0.007090  0.030914
2022-12-27 -0.019374  0.001875  0.002572  0.003504 -0.008208  0.012570
2022-12-28 -0.011064  0.007360 -0.011953  0.005463  0.000932 -0.014751


In [2]:

vine = VineCopula(n_jobs=-1, log_transform=True, random_state=0)
vine.fit(X)
vine.display_vine()

Root Nodes
----------
Node(0): JohnsonSU(a=-0.046, b=1.2, loc=-0.0012, scale=0.033)
Node(1): StudentT(loc=0.00035, scale=0.013, df=2.5)
Node(2): JohnsonSU(a=0.0036, b=1.2, loc=0.00077, scale=0.017)
Node(3): JohnsonSU(a=-0.012, b=1.1, loc=0.00011, scale=0.015)
Node(4): StudentT(loc=0.00039, scale=0.012, df=3.7)
Node(5): StudentT(loc=0.00054, scale=0.011, df=4)

Tree(level 0)
-------------
Edge((0, 2), StudentTCopula(rho=0.316, dof=5.67))
Edge((1, 3), StudentTCopula(rho=0.748, dof=2.66))
Edge((2, 3), StudentTCopula(rho=0.443, dof=4.06))
Edge((2, 4), StudentTCopula(rho=0.332, dof=4.28))
Edge((3, 5), StudentTCopula(rho=0.377, dof=4.61))

Tree(level 1)
-------------
Edge((0, 3) | {2}, StudentTCopula(rho=0.203, dof=8.13))
Edge((1, 5) | {3}, StudentTCopula(rho=0.168, dof=17.51))
Edge((3, 4) | {2}, StudentTCopula(rho=0.215, dof=8.62))
Edge((2, 5) | {3}, StudentTCopula(rho=0.150, dof=8.21))

Tree(level 2)
-------------
Edge((0, 5) | {2, 3}, StudentTCopula(rho=0.100, dof=16.02))
Edge((1, 2) | {3

In [3]:
score = vine.score(X)
aic = vine.aic(X)
print(f"Total Log-likelihood: {score:,.02f}")
print(f"AIC: {aic:,.02f}")

Total Log-likelihood: 132,740.11
AIC: -265,382.23


In [4]:
samples = vine.sample(n_samples=10000)
print(samples.shape)

(10000, 6)


In [5]:
fig = vine.plot_scatter_matrix(X=X)
fig.update_layout(height=600)
show(fig)

In [6]:
amd_dist = vine.marginal_distributions_[0]
amd_dist.plot_pdf(X[["AMD"]])

In [7]:
amd_dist.qq_plot(X[["AMD"]])

In [8]:
edge = vine.trees_[0].edges[0]
copula = edge.copula
print(edge)
print(f"Lower Tail Dependence: {copula.lower_tail_dependence:.2%}")
print(f"Upper Tail Dependence: {copula.upper_tail_dependence:.2%}")

Edge((0, 2), StudentTCopula(rho=0.316, dof=5.67))
Lower Tail Dependence: 10.70%
Upper Tail Dependence: 10.70%


In [9]:
U = compute_pseudo_observations(X[["AMD", "HD"]])
copula.plot_tail_concentration(U)

In [10]:
vine = VineCopula(n_jobs=-1, log_transform=True, central_assets=["AMD"])
vine.fit(X)
vine.display_vine()

Root Nodes
----------
Node(0): JohnsonSU(a=-0.046, b=1.2, loc=-0.0012, scale=0.033)
Node(1): StudentT(loc=0.00035, scale=0.013, df=2.5)
Node(2): JohnsonSU(a=0.0036, b=1.2, loc=0.00077, scale=0.017)
Node(3): JohnsonSU(a=-0.012, b=1.1, loc=0.00011, scale=0.015)
Node(4): StudentT(loc=0.00039, scale=0.012, df=3.7)
Node(5): StudentT(loc=0.00054, scale=0.011, df=4)

Tree(level 0)
-------------
Edge((0, 1), StudentTCopula(rho=0.293, dof=5.56))
Edge((0, 2), StudentTCopula(rho=0.316, dof=5.67))
Edge((0, 3), StudentTCopula(rho=0.307, dof=5.28))
Edge((0, 4), StudentTCopula(rho=0.193, dof=6.09))
Edge((0, 5), StudentTCopula(rho=0.222, dof=7.01))

Tree(level 1)
-------------
Edge((1, 3) | {0}, StudentTCopula(rho=0.726, dof=2.99))
Edge((2, 3) | {0}, StudentTCopula(rho=0.394, dof=5.36))
Edge((2, 4) | {0}, StudentTCopula(rho=0.296, dof=5.61))
Edge((3, 5) | {0}, StudentTCopula(rho=0.342, dof=5.78))

Tree(level 2)
-------------
Edge((1, 5) | {0, 3}, StudentTCopula(rho=0.160, dof=21.56))
Edge((3, 4) | {0,

In [11]:
cond_samples = vine.sample(n_samples=1000, conditioning={"AMD": -0.2})
print(cond_samples.shape)

(1000, 6)


In [12]:
vine = VineCopula(
    n_jobs=-1, log_transform=True, central_assets=["HD", "JPM"], random_state=0
)
vine.fit(X)

conditioning = {"HD": (-0.15, -0.10), "JPM": (None, -0.05)}
cond_samples = vine.sample(n_samples=1000, conditioning=conditioning)
print(cond_samples.shape)

(1000, 6)


In [13]:
vine.plot_marginal_distributions(X=X, conditioning=conditioning)

In [14]:
fig = vine.plot_scatter_matrix(X, conditioning=conditioning)
fig.update_layout(height=600)

In [15]:
model = MeanRisk(risk_measure=RiskMeasure.CVAR)
model.fit(X)
print(model.weights_)

ptf = model.predict(X)

[8.65356305e-12 4.06326174e-12 2.14436665e-01 1.82085950e-11
 3.65721255e-01 4.19842080e-01]


In [16]:
stressed_X = vine.sample(n_samples=50_000, conditioning={"JPM": -0.10})
stressed_ptf = model.predict(stressed_X)

ptf.name = "Unstressed Ptf"
stressed_ptf.name = "Stressed Ptf"
population = Population([ptf, stressed_ptf])
summary = population.summary()
summary.loc[
    ["Mean", "Standard Deviation", "CVaR at 95%", "EVaR at 95%", "Worst Realization"]
]

,Unstressed Ptf,Stressed Ptf
Mean,0.066%,-2.42%
Standard Deviation,1.29%,2.50%
CVaR at 95%,2.83%,8.16%
EVaR at 95%,6.24%,11.23%
Worst Realization,13.77%,22.92%


In [17]:
population.plot_returns_distribution(percentile_cutoff=0.1)

In [18]:
for tree in vine.trees_:
    for edge in tree.edges:
        if isinstance(edge.copula, StudentTCopula):
            edge.copula.rho_ *= 1.1
            edge.copula.dof_ *= 0.8

samples = vine.sample(n_samples=1000)

Conclusion
The flexibility, interpretability, and explicit modeling of tail dependencies make vine copulas an attractive choice for financial applications. 